# Semi-Parametric Regression: Simulation Benchmark (Table 2)
This notebook replicates the simulation study evaluating the proposed **Balanced Penalized Weighted (BPW)** semi-parametric estimators against standard hybrid baselines under severe multicollinearity.


In [8]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.data_loader import generate_simulation_data
from src.hybrid_model import HybridBPWNN
from src.metrics import calculate_metrics, aggregate_cv_results


### 1. Generate Synthetic Data
- $X_1 \sim \mathcal{N}(2, 4)$
- $X_2 = 2 X_1 + \mathcal{N}(0, 4)$ (severe collinearity with $X_1$)
- $X_3 = -0.5 X_1 + X_2 + \mathcal{N}(0, 1)$
- $X_4 \sim \mathcal{U}(-2, 2)$
- $y = 2.2 X_1 - 0.8 X_2 - X_4^2 + \varepsilon$


In [9]:
X, y, df = generate_simulation_data(n=500, seed=42)
print(df.head())


         X1         X2        X3        X4    target
0  2.993428   7.839212  7.741853 -0.920717 -2.104423
1  1.723471   7.265776  7.328674  1.300198 -4.838265
2  3.295377   3.793619  2.205561 -0.006977  3.020961
3  5.046060  11.218058  8.048091 -1.691767 -0.592247
4  1.531693   1.762101  1.694478 -1.765796  0.574523


### 2. Run Monte Carlo Simulation across 100 Repetitions
We compare Ridge-NN, Lasso-NN, Elastic-NN with RBPW-NN, LBPW-NN, and EBPW-NN.


In [10]:
n_repetitions = 100
models_to_run = [
    ('Ridge-NN', 'Ridge'),
    ('Lasso-NN', 'Lasso'),
    ('Elastic-NN', 'ElasticNet'),
    ('RBPW-NN', 'RBPW'),
    ('LBPW-NN', 'LBPW'),
    ('EBPW-NN', 'EBPW'),
]

results = {name: [] for name, _ in models_to_run}

for rep in range(1, n_repetitions + 1):
    X_rep, y_rep, _ = generate_simulation_data(n=500, seed=42 + rep)
    X_p = X_rep[:, [0, 1, 2]]
    X_np = X_rep[:, 3]

    X_tr_p, X_te_p, X_tr_np, X_te_np, y_tr, y_te = train_test_split(
        X_p, X_np, y_rep, test_size=0.2, random_state=42 + rep
    )

    for display_name, m_type in models_to_run:
        hybrid = HybridBPWNN(model_type=m_type, hidden_layers=1, hidden_nodes=50, epochs=100, verbose=0)
        _, y_pred_te, num_params = hybrid.fit_and_predict(X_tr_p, X_te_p, X_tr_np, X_te_np, y_tr)
        te_mse, te_r2, te_aic, te_bic = calculate_metrics(y_te, y_pred_te, num_params)
        results[display_name].append((0, 0, 0, 0, te_mse, te_r2, te_aic, te_bic))

summary = aggregate_cv_results(results)


ValueError: Unknown model_type: RBPW